<a href="https://colab.research.google.com/github/MarilynMaika/Lamhealth-ST/blob/main/01_prepare_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
ROOT = "/content/drive/MyDrive/lamhealth"      # the project folder in Drive
RAW  = f"{ROOT}/data/raw"

for folder in sorted(os.listdir(RAW)):
    files = sorted(os.listdir(f"{RAW}/{folder}"))
    print(f"{folder}: {len(files)} files")
    for f in files:
        print("   ", f)

Mounted at /content/drive
1_original_english: 1 files
    medical_transcripts.csv
2_localised_english: 22 files
    C001_preclampsia_1.csv
    C002_malaria_1.csv
    C003_Diarrhea_1.csv
    C004_Chest_pain_1.csv
    C005_Hypertension_1.csv
    C006_Antenatal_1.csv
    C007_T2DM.csv
    C007_T2DM__1.csv
    C008_PUD_1.csv
    C009_Stroke_1.csv
    C010_Asthma_1.csv
    C013_Appendicitis_1.csv
    C014_Hyperthyroidism_1.csv
    C020_oesteoarthritis_1.csv
    C022_convulsion_1.csv
    C023 PCOS.csv
    C023 PCOS_1.csv
    CO16_HG_1.csv
    Co17_GroinSwelling_1.csv
    c018_anklefracture_1.csv
    c019_pneumonia_1.csv
    c021 meningitis_1.csv
3_lamnso: 12 files
    ANTENATAL FINAL.xlsx
    ASTHMA.xlsx
    C007 T2DM.xlsx
    C008 PUD.xlsx
    C013 APENDICITIS.xlsx
    C09 STROKE 1.xlsx
    C14 HYPERTHYROISM.xlsx
    CHEST PAINS.xlsx
    DIARRHEA.xlsx
    HYPERTENTION.xlsx
    MALARIA.xlsx
    PRECLAMPSIA.xlsx
4_parallel: 11 files
    ASTHMA_pd.xlsx
    C007 T2DM_pd.xlsx
    C008 PUD_pd.xls

In [2]:
import hashlib
import pandas as pd

def sha256(path):
    """Return the SHA-256 fingerprint of a file's exact bytes."""
    with open(path, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()

ENG = f"{RAW}/2_localised_english"
pairs = [("C007_T2DM.csv", "C007_T2DM__1.csv"),
         ("C023 PCOS.csv", "C023 PCOS_1.csv")]

for a, b in pairs:
    ha, hb = sha256(f"{ENG}/{a}"), sha256(f"{ENG}/{b}")
    print(f"{a}  {ha[:12]}…")
    print(f"{b}  {hb[:12]}…")
    if ha == hb:
        print("   → identical files\n")
    else:
        da = pd.read_csv(f"{ENG}/{a}", encoding="utf-8-sig")
        db = pd.read_csv(f"{ENG}/{b}", encoding="utf-8-sig")
        print(f"   → DIFFERENT: {a} has {len(da)} rows, {b} has {len(db)} rows")
        print(f"     columns: {list(da.columns)}  vs  {list(db.columns)}\n")

C007_T2DM.csv  8475c26a65c6…
C007_T2DM__1.csv  8475c26a65c6…
   → identical files

C023 PCOS.csv  36225938253b…
C023 PCOS_1.csv  a36dfc90aa97…
   → DIFFERENT: C023 PCOS.csv has 102 rows, C023 PCOS_1.csv has 102 rows
     columns: ['ID', 'Speaker', 'Utterance']  vs  ['ID', 'Speaker', 'Utterance', 'Unnamed: 3']



In [3]:
a = pd.read_csv(f"{ENG}/C023 PCOS.csv",   encoding="utf-8-sig")
b = pd.read_csv(f"{ENG}/C023 PCOS_1.csv", encoding="utf-8-sig")

print("Extra column is completely empty:", b["Unnamed: 3"].isna().all())
print("Same text in the shared columns: ", a.equals(b[a.columns]))

Extra column is completely empty: False
Same text in the shared columns:  True


In [6]:
cols = ["consultation_id", "condition", "localised_english", "lamnso", "parallel", "status", "note"]
inventory = pd.DataFrame([
    ("C001", "Pre-eclampsia",         "C001_preclampsia_1.csv",      "PRECLAMPSIA.xlsx",      "PRECLAMPSIA_pd.csv",       "parallel_ready", ""),
    ("C002", "Malaria",               "C002_malaria_1.csv",          "MALARIA.xlsx",          "MALARIA_pd.csv",           "parallel_ready", ""),
    ("C003", "Diarrhoea",             "C003_Diarrhea_1.csv",         "DIARRHEA.xlsx",         "DIARRHEA_pd.csv",          "parallel_ready", ""),
    ("C004", "Chest pain",            "C004_Chest_pain_1.csv",       "CHEST PAINS.xlsx",      "CHEST PAINS_pd.csv",       "parallel_ready", ""),
    ("C005", "Hypertension",          "C005_Hypertension_1.csv",     "HYPERTENTION.xlsx",     "",                         "excluded",       "unresolved translation issues"),
    ("C006", "Antenatal visit",       "C006_Antenatal_1.csv",        "ANTENATAL FINAL.xlsx",  "antenatal_pd.csv",         "parallel_ready", ""),
    ("C007", "Type 2 diabetes",       "C007_T2DM__1.csv",            "C007 T2DM.xlsx",        "C007 T2DM_pd.xlsx",        "parallel_ready", "C007_T2DM.csv is an identical copy"),
    ("C008", "Peptic ulcer disease",  "C008_PUD_1.csv",              "C008 PUD.xlsx",         "C008 PUD_pd.xlsx",         "parallel_ready", ""),
    ("C009", "Stroke",                "C009_Stroke_1.csv",           "C09 STROKE 1.xlsx",     "C09 STROKE_pd.xlsx",       "parallel_ready", ""),
    ("C010", "Asthma",                "C010_Asthma_1.csv",           "ASTHMA.xlsx",           "ASTHMA_pd.xlsx",           "parallel_ready", ""),
    ("C013", "Appendicitis",          "C013_Appendicitis_1.csv",     "C013 APENDICITIS.xlsx", "C013 APENDICITIS_pd.xlsx", "parallel_ready", ""),
    ("C014", "Hyperthyroidism",       "C014_Hyperthyroidism_1.csv",  "C14 HYPERTHYROISM.xlsx","C14 HYPERTHYROISM_pd.xlsx","parallel_ready", ""),
    ("C016", "Hyperemesis gravidarum","CO16_HG_1.csv",               "", "", "english_only", ""),
    ("C017", "Groin swelling",        "Co17_GroinSwelling_1.csv",    "", "", "english_only", ""),
    ("C018", "Ankle fracture",        "c018_anklefracture_1.csv",    "", "", "english_only", ""),
    ("C019", "Pneumonia",             "c019_pneumonia_1.csv",        "", "", "english_only", ""),
    ("C020", "Osteoarthritis",        "C020_oesteoarthritis_1.csv",  "", "", "english_only", ""),
    ("C021", "Meningitis",            "c021 meningitis_1.csv",       "", "", "english_only", ""),
    ("C022", "Convulsion",            "C022_convulsion_1.csv",       "", "", "english_only", ""),
    ("C023", "PCOS",                  "C023 PCOS.csv",               "", "", "english_only", "C023 PCOS_1.csv: same text plus a stray '-' in an extra column (row T098)"),
], columns=cols)

folders = {"localised_english": "2_localised_english", "lamnso": "3_lamnso", "parallel": "4_parallel"}

# 1. Does every file named in the inventory exist?
missing = [(r.consultation_id, f"{folders[c]}/{r[c]}")
           for _, r in inventory.iterrows() for c in folders
           if r[c] and not os.path.exists(f"{RAW}/{folders[c]}/{r[c]}")]
print("Listed but not found:", missing or "none")

# 2. Is any file in the folders left out of the inventory?
for c, folder in folders.items():
    unlisted = set(os.listdir(f"{RAW}/{folder}")) - set(inventory[c])
    print(f"Not in inventory ({folder}):", sorted(unlisted) or "none")

print()
print(inventory["status"].value_counts().to_string())

Listed but not found: none
Not in inventory (2_localised_english): ['C007_T2DM.csv', 'C023 PCOS_1.csv']
Not in inventory (3_lamnso): none
Not in inventory (4_parallel): none

status
parallel_ready    11
english_only       8
excluded           1


In [7]:
os.makedirs(f"{ROOT}/data/manifests", exist_ok=True)
inventory.to_csv(f"{ROOT}/data/manifests/inventory.csv", index=False)
print("Saved", len(inventory), "rows")

Saved 20 rows


## Step 4: Reading the 11 parallel files

In [9]:
def read_any(path):
    """Read a CSV or Excel file, keeping every cell as text, exactly as stored."""
    if path.lower().endswith(".csv"):
        return pd.read_csv(path, encoding="utf-8-sig", dtype=str, keep_default_na=False)
    return pd.read_excel(path, dtype=str, keep_default_na=False)

ready = inventory[inventory["status"] == "parallel_ready"]
frames = {}

for _, r in ready.iterrows():
    path = f"{RAW}/4_parallel/{r.parallel}"
    try:
        df = read_any(path)
    except UnicodeDecodeError as e:
        print(f"{r.consultation_id}: NOT UTF-8, stopped -> {e}")
        continue
    frames[r.consultation_id] = df
    print(f"{r.consultation_id} {r.condition}: {len(df)} rows")
    print(f"   columns: {list(df.columns)}")
    lam_cols = [c for c in df.columns if "lamnso" in c.lower()]
    if lam_cols:
        print(f"   sample:  {df[lam_cols[0]].iloc[1]}")
    print()

C001 Pre-eclampsia: 82 rows
   columns: ['ID', 'Speaker', 'Lamnso Utterance', 'English Utterance']
   sample:  Yì ràn ne à ŋgàywìr

C002 Malaria: 60 rows
   columns: ['Turn', 'Speaker', 'Lamnso Utterance', 'English Utterance']
   sample:  À rə̀sí ne à Tàádòm

C003 Diarrhoea: 107 rows
   columns: ['Turn', 'Speaker', 'Lamnso Utterance', 'English Utterance']
   sample:  À rə̀sí ne à. Yír yémé dzə̀ Taata.

C004 Chest pain: 75 rows
   columns: ['Turn', 'Speaker', 'Lamnso Utterance', 'English Utterance']
   sample:  M ta’ e kfə̀n kin kuumé yír yé

C006 Antenatal visit: 86 rows
   columns: ['Turn', 'Speaker', 'Lamnso Utterance', 'English Utterance']
   sample:  Aa  Tàádòm, m kér kishérì fóo yen wò.

C007 Type 2 diabetes: 60 rows
   columns: ['Turn', 'Speaker', 'Lamnso Utterance', 'English Utterance']
   sample:  M lo e kiíy yír yé wun à viyá vé. 

C008 Peptic ulcer disease: 60 rows
   columns: ['Turn', 'Speaker', 'English Utterance', 'Utterance']

C009 Stroke

In [10]:
for cid in ["C008", "C014"]:
    print(cid, list(frames[cid].columns))
    print(frames[cid].head(4).to_string())
    print()

C008 ['Turn', 'Speaker', 'English Utterance', 'Utterance']
   Turn       Speaker                                              English Utterance                                                 Utterance
0  T001     Tàádòm  Èé à rə́sí kijuŋ a. Wiy kijuŋ e la’ shìv. M dzə̀ Joseph.  Hello, good evening. Welcome to the clinic. I am Joseph.
1  T002  Ŋgaà Yàsí                                             À rə́sí kijuŋ a.                                             Good evening.
2  T003     Tàádòm                                          À dzə̀ viyá visì’?                                          How old are you?
3  T004  Ŋgaà Yàsí                                 Kiyá nshòv sitàn e mtaár.                                              35 years old.

C014 ['Turn', 'Speaker', 'English Utterance', 'Lamnso Utterance']
   Turn       Speaker                                        English Utterance                                                   Lamnso Utterance
0  T001     T

In [11]:
import unicodedata

# Which column holds what, per file. Files not listed use the common names.
DEFAULT = {"turn": "Turn", "speaker": "Speaker",
           "lamnso": "Lamnso Utterance", "english": "English Utterance"}
COLUMN_MAP = {
    "C001": {"turn": "ID"},
    "C008": {"lamnso": "English Utterance", "english": "Utterance"},          # headers mislabelled
    "C014": {"lamnso": "English Utterance", "english": "Lamnso Utterance"},   # headers swapped
}

def looks_lamnso(text):
    """True if the text has a letter or tone mark English almost never uses."""
    if any(ch in "əŋƏŊ" for ch in text):
        return True
    return any(unicodedata.combining(c) for c in unicodedata.normalize("NFD", text))

parts = []
for cid, df in frames.items():
    m = {**DEFAULT, **COLUMN_MAP.get(cid, {})}
    std = pd.DataFrame({new: df[old] for new, old in m.items()})
    std.insert(0, "consultation_id", cid)
    parts.append(std)

    filled = std[(std["lamnso"].str.strip() != "") & (std["english"].str.strip() != "")]
    lam_ok = filled["lamnso"].map(looks_lamnso).mean() * 100
    eng_bad = filled["english"].map(looks_lamnso).mean() * 100
    flag = "  <-- CHECK" if lam_ok < 70 or eng_bad > 10 else ""
    print(f"{cid}: Lamnso column looks like Lamnso in {lam_ok:5.1f}% of rows | "
          f"English column looks like Lamnso in {eng_bad:4.1f}%{flag}")

combined = pd.concat(parts, ignore_index=True)
print(f"\nCombined: {len(combined)} rows from {combined['consultation_id'].nunique()} consultations")
print("\nSpeaker labels used:")
print(combined["speaker"].value_counts().to_string())

C001: Lamnso column looks like Lamnso in  98.8% of rows | English column looks like Lamnso in  0.0%
C002: Lamnso column looks like Lamnso in 100.0% of rows | English column looks like Lamnso in  0.0%
C003: Lamnso column looks like Lamnso in  96.3% of rows | English column looks like Lamnso in  0.9%
C004: Lamnso column looks like Lamnso in  97.3% of rows | English column looks like Lamnso in  0.0%
C006: Lamnso column looks like Lamnso in 100.0% of rows | English column looks like Lamnso in  0.0%
C007: Lamnso column looks like Lamnso in 100.0% of rows | English column looks like Lamnso in  0.0%
C008: Lamnso column looks like Lamnso in 100.0% of rows | English column looks like Lamnso in  0.0%
C009: Lamnso column looks like Lamnso in 100.0% of rows | English column looks like Lamnso in  0.0%
C010: Lamnso column looks like Lamnso in 100.0% of rows | English column looks like Lamnso in  0.0%
C013: Lamnso column looks like Lamnso in  98.4% of rows | English column looks like Lamnso in  0.0%


## Step 7: Give each row a clean speaker role

In [12]:
def speaker_role(label):
    """Map every spelling of the two speakers to Doctor or Patient."""
    k = unicodedata.normalize("NFC", label).strip().strip(".").lower().replace("ŋ", "n")
    if k in {"tàádòm", "tàácdòm", "doctor"}:
        return "Doctor"
    if k in {"ngaà yàsí", "patient"}:
        return "Patient"
    return ""          # empty or unrecognised: we look at these by hand

combined["role"] = combined["speaker"].map(speaker_role)

# Every original label, shown with its length so hidden spaces are visible
labels = combined["speaker"].map(lambda s: f"{s!r} (len {len(s)})")
print(pd.crosstab(labels, combined["role"]).to_string())

print("\nRows with no role:")
print(combined.loc[combined["role"] == "", ["consultation_id", "turn", "speaker", "english"]].to_string())

print("\nC003 English text that looks like Lamnso:")
c3 = combined[(combined["consultation_id"] == "C003") & combined["english"].map(looks_lamnso)]
print(c3[["turn", "lamnso", "english"]].to_string())

role                         Doctor  Patient
speaker                                     
'' (len 0)                3       0        0
'Doctor' (len 6)          0       4        0
'Ngaà Yàsí' (len 12)   0       0       37
'Patient' (len 7)         0       0        5
'Tàácdòm' (len 10)     0       1        0
'Tàádòm ' (len 10)     0       5        0
'Tàádòm' (len 9)       0     386        0
'Tàádòm. ' (len 11)    0       1        0
'Ŋgaà Yàsí ' (len 13)  0       0        4
'Ŋgaà Yàsí' (len 12)   0       0      268
'Ŋgaà yàsí' (len 12)   0       0       50

Rows with no role:
    consultation_id  turn speaker                                                                                                                                                                                                                                                                                                               english
76             C001  T077          Before starting 

In [13]:
import re

# 1. Speaker for the three unlabelled rows, decided by reading the English text
ROLE_FIX = {("C001", "T077"): "Doctor",
            ("C006", "T077"): "Doctor",
            ("C006", "T080"): "Doctor"}
for (cid, turn), role in ROLE_FIX.items():
    mask = (combined["consultation_id"] == cid) & (combined["turn"] == turn)
    assert mask.sum() == 1, f"{cid} {turn} matched {mask.sum()} rows"
    combined.loc[mask, "role"] = role
print(combined["role"].value_counts().to_string())

# 2. Scan both text columns for typical signs of a wrong-encoding save
GARBLED = re.compile(r"â€|Ã|Â|\ufffd")
print("\nGarbled characters found:")
for col in ["lamnso", "english"]:
    hits = combined[combined[col].str.contains(GARBLED)]
    for _, r in hits.iterrows():
        bad = GARBLED.search(r[col])
        start = max(0, bad.start() - 25)
        print(f"  {r.consultation_id} {r.turn} [{col}]: ...{r[col][start:bad.end() + 25]}...")

role
Doctor     400
Patient    364

Garbled characters found:
  C003 T058 [english]: ...ame to the hospital todayâ€”to find out what's wrong...


## Step 9: Normalise the text and check for problems

In [14]:
clean = combined.copy()

# 1. Explicit repairs, one entry per damaged cell
FIXES = {("C003", "T058", "english"): ("â€”", "—")}
for (cid, turn, col), (old, new) in FIXES.items():
    mask = (clean["consultation_id"] == cid) & (clean["turn"] == turn)
    assert mask.sum() == 1 and old in clean.loc[mask, col].iloc[0], f"fix not applicable: {cid} {turn}"
    clean.loc[mask, col] = clean.loc[mask, col].str.replace(old, new, regex=False)

# 2. NFC + trim the edges, counting what each one changed
for col in ["turn", "speaker", "lamnso", "english"]:
    before = clean[col]
    nfc = before.map(lambda s: unicodedata.normalize("NFC", s))
    clean[col] = nfc.str.strip()
    print(f"{col:8}: {(nfc != before).sum():3} cells re-encoded (NFC), "
          f"{(clean[col] != nfc).sum():3} cells had edge spaces removed")

# 3. Checks
empty = clean[(clean["lamnso"] == "") | (clean["english"] == "")]
print(f"\nRows with an empty Lamnso or English cell: {len(empty)}")
if len(empty):
    print(empty[["consultation_id", "turn", "role", "lamnso", "english"]].to_string())

dups = clean[clean.duplicated(["consultation_id", "turn"], keep=False)]
print(f"\nRepeated turn numbers within a consultation: {len(dups)}")
if len(dups):
    print(dups[["consultation_id", "turn", "english"]].to_string())

odd = clean[~clean["turn"].str.fullmatch(r"T\d{3}")]
print(f"\nTurn labels not in the form T001: {len(odd)}")
if len(odd):
    print(odd[["consultation_id", "turn"]].to_string())

inner = clean["lamnso"].str.contains("  ").sum() + clean["english"].str.contains("  ").sum()
print(f"\nCells with a double space inside (left as written): {inner}")

turn    :   0 cells re-encoded (NFC),   0 cells had edge spaces removed
speaker : 752 cells re-encoded (NFC),  10 cells had edge spaces removed
lamnso  : 749 cells re-encoded (NFC), 244 cells had edge spaces removed
english :   0 cells re-encoded (NFC),   5 cells had edge spaces removed

Rows with an empty Lamnso or English cell: 1
   consultation_id  turn    role lamnso                                                                                                                                                                                                                                                                                                               english
76            C001  T077  Doctor         Before starting treatment, we will perform several laboratory tests, including a full blood count, electrolytes, urea, creatinine, uric acid, liver function tests, and urinalysis to check for protein in your urine. We will also repeat your ultrasound scan and continue monito

## Step 10: Look at the unnumbered rows in context

In [15]:
def context(cid, rows_before=2, rows_after=2):
    sub = clean[clean["consultation_id"] == cid].reset_index()
    blanks = sub.index[sub["turn"] == ""]
    show = sorted({i for b in blanks for i in range(b - rows_before, b + rows_after + 1)
                   if 0 <= i < len(sub)})
    print(sub.loc[show, ["turn", "role", "english"]].to_string())
    print()

context("C006")
context("C010")

    turn     role                                                                    english
27  T028  Patient  Yes. I feel a burning or peppery sensation just before I start urinating.
28  T029   Doctor                         That is not normal. We will investigate the cause.
29         Doctor                                                Have you been having fever?
30  T030  Patient                            Sometimes I feel warm at night, but it is mild.
31  T031   Doctor                        Have you noticed any vaginal discharge or bleeding?

    turn     role                                                                                                                                                       english
58  T059  Patient                      When I eat nuts, I notice swelling around my face sometimes, and then I notice also this rash around my arm, my face, my body generally.
59  T060   Doctor  Oh, I see. Okay. So, I was asking if you know anybody in your family w

In [16]:
DATA_VERSION = "v1"
PROC, MAN = f"{ROOT}/data/processed", f"{ROOT}/data/manifests"
os.makedirs(PROC, exist_ok=True)
out = f"{PROC}/parallel_{DATA_VERSION}.csv"
assert not os.path.exists(out), f"{out} already exists: data {DATA_VERSION} is frozen. Use a new version."

# 1. IDs from position in the raw file (clean is still in file order)
clean["row_in_file"] = clean.groupby("consultation_id").cumcount() + 1
clean["pair_id"] = clean["consultation_id"] + "_" + clean["row_in_file"].map("{:03d}".format)

# 2. Drop rows with an empty side, and record which and why (IDs only, no text)
gap = (clean["lamnso"] == "") | (clean["english"] == "")
dropped = clean.loc[gap, ["pair_id"]].assign(reason="empty Lamnso or English cell")
data = clean.loc[~gap, ["pair_id", "consultation_id", "turn", "role", "speaker", "lamnso", "english"]].reset_index(drop=True)

# 3. Save, then read it back and confirm nothing changed on the way to disk
data.to_csv(out, index=False, encoding="utf-8")
check = pd.read_csv(out, encoding="utf-8", dtype=str, keep_default_na=False)
assert check.equals(data), "File on disk differs from the data in memory"
print(f"Saved {len(data)} pairs to {out} and read back identically")

# 4. Fingerprints of the 11 raw files used and of the frozen file
ready = inventory[inventory["status"] == "parallel_ready"]
manifest = [{"stage": "raw_parallel", "consultation_id": r.consultation_id, "file": r.parallel,
             "sha256": sha256(f"{RAW}/4_parallel/{r.parallel}")} for _, r in ready.iterrows()]
manifest.append({"stage": "processed", "consultation_id": "ALL", "file": os.path.basename(out),
                 "sha256": sha256(out)})
manifest = pd.DataFrame(manifest)

# 5. Statistics for the report
stats = data.assign(lamnso_words=data["lamnso"].str.split().str.len(),
                    english_words=data["english"].str.split().str.len()) \
            .groupby("consultation_id") \
            .agg(pairs=("pair_id", "size"),
                 lamnso_words_mean=("lamnso_words", "mean"),
                 english_words_mean=("english_words", "mean")).round(1)

manifest.to_csv(f"{MAN}/manifest_{DATA_VERSION}.csv", index=False)
dropped.to_csv(f"{MAN}/dropped_{DATA_VERSION}.csv", index=False)
stats.to_csv(f"{MAN}/stats_{DATA_VERSION}.csv")

print(dropped.to_string(index=False))
print(stats.to_string())
print(f"\nTotal: {len(data)} pairs | processed sha256: {manifest.iloc[-1]['sha256'][:16]}…")

Saved 763 pairs to /content/drive/MyDrive/lamhealth/data/processed/parallel_v1.csv and read back identically
 pair_id                       reason
C001_077 empty Lamnso or English cell
                 pairs  lamnso_words_mean  english_words_mean
consultation_id                                              
C001                81               14.0                 9.1
C002                60                9.4                 8.6
C003               107               12.5                11.0
C004                75                9.3                 7.4
C006                86               14.5                 9.4
C007                60               10.2                 8.4
C008                60               11.1                10.8
C009                60               13.6                12.4
C010                64               10.5                10.4
C013                62                9.6                 9.9
C014                48                8.8                 9.5

Total: 7

##12: Choose and lock the test consultations

In [17]:
import numpy as np

SPLIT_SEED, N_TEST = 2026, 2
consultations = sorted(data["consultation_id"].unique())
test_cons = sorted(np.random.RandomState(SPLIT_SEED).choice(consultations, N_TEST, replace=False))
print("Test consultations:", test_cons,
      "->", inventory.set_index("consultation_id").loc[test_cons, "condition"].tolist())

def match_key(s):
    """Only for spotting repeated sentences; the stored text is not changed."""
    s = s.replace("\u2019", "'").replace("\u2018", "'").lower()
    return re.sub(r"\s+", " ", s).strip(" .?!,")

split = pd.DataFrame({"pair_id": data["pair_id"],
                      "split": np.where(data["consultation_id"].isin(test_cons), "test", "train")})
train_keys = set(data.loc[split["split"] == "train", "lamnso"].map(match_key))
repeat = (split["split"] == "test") & data["lamnso"].map(match_key).isin(train_keys)
split.loc[repeat, "split"] = "test_excluded_repeat"

print(split["split"].value_counts().to_string())
print("\nTest sentences already seen in training (not scored):")
print(data.loc[repeat, ["pair_id", "english"]].to_string(index=False))

split_path = f"{MAN}/split_{DATA_VERSION}.csv"
assert not os.path.exists(split_path), "Split already locked."
split.to_csv(split_path, index=False)
pd.concat([manifest, pd.DataFrame([{"stage": "split", "consultation_id": ",".join(test_cons),
                                    "file": os.path.basename(split_path), "sha256": sha256(split_path)}])]) \
  .to_csv(f"{MAN}/manifest_{DATA_VERSION}.csv", index=False)
print("\nSplit locked and fingerprinted.")

Test consultations: [np.str_('C003'), np.str_('C004')] -> ['Diarrhoea', 'Chest pain']
split
train                   581
test                    162
test_excluded_repeat     20

Test sentences already seen in training (not scored):
 pair_id                          english
C003_046                              No.
C003_048                              No.
C003_055   Do you drink alcohol or smoke?
C003_076                      No, Doctor.
C003_077          Do you have chest pain?
C003_078                  No, not at all.
C003_086                              No.
C003_090                             Yes.
C003_101                             Yes.
C004_006               How are you today?
C004_008     What's your complaint today?
C004_027                      No, Doctor.
C004_029                      No, Doctor.
C004_031                              No.
C004_046 Have you had any surgery before?
C004_047                               No
C004_049                      No, Doctor.
C004_053     